# Anma, Checkpoint 3: pembersihan dan analisis celah
Pertanyaan: seperti apa pola tontonan saya dibanding katalog populer MAL, dan franchise populer mana yang belum tersentuh?
Satuan analisis utama adalah **franchise**, bukan judul, karena sequel, season, OVA, dan spin-off pendek dihitung terpisah di MAL.
Batasan data: tidak ada rating pribadi. Status `completed` dipakai sebagai sinyal suka tersirat.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
AKSEN, ABU = "#e0a800", "#b8b8b8"
os.makedirs("gambar", exist_ok=True)

saya = pd.read_csv("data/list_saya.csv")
kat = pd.read_csv("data/katalog_populer.csv")
print("List saya:", saya.shape, "| Katalog populer:", kat.shape)
saya.head(3)

## 1. Pembersihan data

In [ ]:
print("Nilai kosong di list saya:")
print(saya.isna().sum()[lambda s: s > 0])
print("\nNilai kosong di katalog:")
print(kat.isna().sum()[lambda s: s > 0])
print("\nDuplikat id:", saya["id"].duplicated().sum(), "(list),", kat["id"].duplicated().sum(), "(katalog)")

**Judul panjang atau masih tayang** (mis. One Piece) ditandai terpisah supaya tidak membiaskan hasil.
Aturannya: 100 episode atau lebih, atau berstatus `currently_airing`.

In [ ]:
def panjang(df):
    return (df["episode"].fillna(0) >= 100) | (df["status_tayang"] == "currently_airing")

kat["panjang"] = panjang(kat)
saya["panjang"] = panjang(saya)
print("Judul panjang atau masih tayang di katalog:", int(kat["panjang"].sum()))
kat[kat["panjang"]].nsmallest(10, "peringkat_populer")[["judul", "episode", "status_tayang", "peringkat_populer"]]

**Cek kualitas pengelompokan franchise.** Lihat franchise terbesar dan anggotanya.
Pertanyaan untuk Anda: apakah anggotanya benar satu franchise? Catat yang janggal (dua franchise berbeda yang tercampur).

In [ ]:
besar = kat.groupby("franchise_id").agg(nama=("franchise_nama", "first"), jumlah=("id", "count")).nlargest(8, "jumlah")
display(besar)
for fid in besar.index[:3]:
    print(besar.loc[fid, "nama"], "->", ", ".join(kat.loc[kat["franchise_id"] == fid, "judul"].head(12)))

**Tabel franchise.** Satu baris per franchise di katalog, diwakili judul yang paling populer.
`selesai` = ada judul franchise itu berstatus completed di list saya. `pernah` = ada judul selain plan_to_watch.

In [ ]:
wakil = kat.sort_values("peringkat_populer").drop_duplicates("franchise_id").set_index("franchise_id")
fr = wakil[["franchise_nama", "peringkat_populer", "jumlah_pengguna_list", "genre", "sumber", "tahun"]].rename(
    columns={"peringkat_populer": "peringkat_terbaik"})
fr["peringkat_terbaik"] = fr["peringkat_terbaik"].clip(upper=2000)
fr["panjang"] = kat.groupby("franchise_id")["panjang"].any()
selesai = set(saya.loc[saya["status_list"] == "completed", "franchise_id"])
pernah = set(saya.loc[saya["status_list"] != "plan_to_watch", "franchise_id"])
fr["selesai"] = fr.index.isin(selesai)
fr["pernah"] = fr.index.isin(pernah)
print(f"Franchise di katalog: {len(fr)} | sudah saya selesaikan: {fr['selesai'].sum()} ({fr['selesai'].mean():.1%})")

## 2. Analisis dan grafik

### Grafik 1: komposisi list saya

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
saya["status_list"].value_counts().plot.barh(ax=ax[0], color=AKSEN)
ax[0].set_title("Status di list saya"); ax[0].invert_yaxis()
saya["tipe"].value_counts().plot.barh(ax=ax[1], color=AKSEN)
ax[1].set_title("Tipe judul di list saya"); ax[1].invert_yaxis()
plt.tight_layout(); plt.savefig("gambar/1_komposisi_list.png", dpi=150, bbox_inches="tight"); plt.show()

### Grafik 2: entri versus franchise

In [ ]:
n_entri, n_fr = len(saya), saya["franchise_id"].nunique()
ukuran = saya.groupby("franchise_nama").size().nlargest(10).sort_values()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1, 2]})
ax[0].bar(["Entri", "Franchise"], [n_entri, n_fr], color=[ABU, AKSEN])
for i, v in enumerate([n_entri, n_fr]):
    ax[0].text(i, v, str(v), ha="center", va="bottom")
ax[0].set_title("List saya: entri vs franchise")
ukuran.plot.barh(ax=ax[1], color=AKSEN)
ax[1].set_title("10 franchise terbesar di list saya (jumlah entri)")
plt.tight_layout(); plt.savefig("gambar/2_entri_vs_franchise.png", dpi=150, bbox_inches="tight"); plt.show()

### Grafik 3: cakupan per pita popularitas
Dari franchise di tiap pita peringkat, berapa persen yang sudah saya selesaikan?

In [ ]:
batas = [0, 100, 250, 500, 1000, 1500, 2000]
fr["pita"] = pd.cut(fr["peringkat_terbaik"], batas, labels=[f"{a + 1}-{b}" for a, b in zip(batas[:-1], batas[1:])])
cakupan = fr.groupby("pita", observed=True).agg(franchise=("selesai", "size"), selesai=("selesai", "sum"))
cakupan["persen"] = 100 * cakupan["selesai"] / cakupan["franchise"]
ax = cakupan["persen"].plot.bar(color=AKSEN, figsize=(8, 3.8), rot=0)
ax.set_ylabel("% franchise yang saya selesaikan"); ax.set_title("Cakupan tontonan per pita peringkat popularitas")
for i, (p, n) in enumerate(zip(cakupan["persen"], cakupan["franchise"])):
    ax.text(i, p, f"{p:.0f}%\n(n={n})", ha="center", va="bottom", fontsize=8)
plt.savefig("gambar/3_cakupan_per_pita.png", dpi=150, bbox_inches="tight"); plt.show()
cakupan

### Grafik 4: celah genre
Genre dengan cakupan terendah dan tertinggi, hanya genre yang punya 30 franchise populer atau lebih.

In [ ]:
g = fr.assign(genre=fr["genre"].fillna("").str.split("|")).explode("genre")
g = g[g["genre"] != ""]
cg = g.groupby("genre").agg(populer=("selesai", "size"), selesai=("selesai", "sum"))
cg = cg[cg["populer"] >= 30].copy()
cg["persen"] = 100 * cg["selesai"] / cg["populer"]
rendah, tinggi = cg.nsmallest(8, "persen"), cg.nlargest(8, "persen")
tampil = pd.concat([rendah, tinggi]).sort_values("persen")
tampil = tampil[~tampil.index.duplicated()]
warna = [AKSEN if i in rendah.index else ABU for i in tampil.index]
ax = tampil["persen"].plot.barh(color=warna, figsize=(7, 5))
ax.set_xlabel("% franchise populer yang saya selesaikan"); ax.set_title("Genre: cakupan terendah (kuning) dan tertinggi (abu)")
plt.savefig("gambar/4_celah_genre.png", dpi=150, bbox_inches="tight"); plt.show()
cg.sort_values("persen").head(10)

### Grafik 5: cakupan menurut sumber cerita dan era tayang

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
s = fr.groupby("sumber").agg(n=("selesai", "size"), selesai=("selesai", "sum"))
s = s[s["n"] >= 20]
(100 * s["selesai"] / s["n"]).sort_values().plot.barh(ax=ax[0], color=AKSEN)
ax[0].set_title("Cakupan per sumber cerita (%)")
e = fr.dropna(subset=["tahun"]).assign(era=lambda d: (d["tahun"] // 5 * 5).astype(int))
e = e.groupby("era").agg(n=("selesai", "size"), selesai=("selesai", "sum"))
e = e[e["n"] >= 15]
(100 * e["selesai"] / e["n"]).plot.bar(ax=ax[1], color=AKSEN, rot=0)
ax[1].set_title("Cakupan per era tayang, 5 tahunan (%)")
plt.tight_layout(); plt.savefig("gambar/5_sumber_dan_era.png", dpi=150, bbox_inches="tight"); plt.show()

### Grafik 6: franchise populer yang belum tersentuh
Batang abu = franchise panjang atau masih tayang (butuh komitmen besar, jadi dipisahkan).

In [ ]:
belum = fr[~fr["pernah"]].nsmallest(15, "peringkat_terbaik").sort_values("peringkat_terbaik", ascending=False)
warna = [ABU if p else AKSEN for p in belum["panjang"]]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(belum["franchise_nama"], belum["jumlah_pengguna_list"] / 1e6, color=warna)
ax.set_xlabel("Jumlah pengguna MAL yang memasukkan ke list (juta)")
ax.set_title("15 franchise populer yang belum pernah saya tonton")
plt.tight_layout(); plt.savefig("gambar/6_belum_tersentuh.png", dpi=150, bbox_inches="tight"); plt.show()
belum.sort_values("peringkat_terbaik")[["franchise_nama", "peringkat_terbaik", "panjang"]]

## 3. Ringkasan angka

In [ ]:
di_luar = (~saya["id"].isin(kat["id"])).sum()
print("Entri di list saya:", len(saya), "| franchise:", saya["franchise_id"].nunique())
print(f"Entri di luar 2000 judul populer: {di_luar} ({di_luar / len(saya):.0%})")
print("Median peringkat popularitas judul di list saya:", saya["peringkat_populer"].median())
print(f"Franchise populer yang sudah saya selesaikan: {fr['selesai'].sum()} dari {len(fr)} ({fr['selesai'].mean():.0%})")
top250 = fr[fr["peringkat_terbaik"] <= 250]
print(f"Dari 250 peringkat teratas: {top250['selesai'].sum()} dari {len(top250)} ({top250['selesai'].mean():.0%})")
print("Franchise populer belum pernah ditonton:", int((~fr['pernah']).sum()), "| di antaranya panjang/masih tayang:", int((~fr['pernah'] & fr['panjang']).sum()))
print("Genre cakupan terendah:", ", ".join(cg.nsmallest(3, "persen").index))

## 4. Temuan saya (isi sendiri, 3 sampai 5 kalimat dengan angka dari bagian 3)
- Dari grafik analisis francise yang sudah ditonton, misal 78 francise dengan rating tertinggi sampai rank 100, 31% baru saya tonton. Kemudian grafik menurun yang menandakan terkadang saya menurut pada rating untuk menonton Anime
- Dari grafik analisis genre, genre SoL terutama Cute Girl Doing Cute Things menempati posisi tertinggi list tontonan saya, yang berarti kemungkinan saya cenderung menonton anime untuk relax, melepas penat menonton anime SoL atau anime CGDCT yang biasanya tidak perlu memikirkan alur cerita atau drama yang ada. Genre Mecha, Sci-fi memang tidak pernah masuk list karena saya pribadi tidak menyukai genre tersebut.
- Dari grafik sumber manga, 4-Koma manga menempati posisi pertama list tontonan saya. Hal ini masuk akal menurut saya karena kemungkinan anime SoL atau anime CGDCT kebanyakan berasal dari 4-Koma. Selain itu anime era 2015-2025 menempati posisi teratas list tontonan saya, kemungkinan karena kualitas anime yang sangat berbeda dengan era 2000 an. Meski tetap ada anime 2000 an yang saya tonton.
- Dari grafik francise populer yang belum ditonton, Kebanyakan adalah anime shounen yang penuh konflik seperti FMA atau cerita yang sama sekali tidak SoL biasa, seperti drama berat yang dibawa Koe no Katachi.

**Keterbatasan:** tidak ada rating pribadi, list relatif kecil, dan "belum ditonton" tidak sama dengan "tidak suka".